# 🔥 Multi-Agent War Room — GRPO Training

**Team ClaudeStalkers** — Siddharth, Lakshminath, Lokesh — BITS Pilani Hyderabad

This notebook trains the Diagnosis agent using GRPO with 4 independent reward functions.
Set runtime to **GPU (T4 or A100)** before running.

In [ ]:
# Cell 1: Clone repo and install dependencies (~3 min)
!git clone https://github.com/Git4Lokesh/Meta_Hackathon_ClaudeStalkers.git
%cd Meta_Hackathon_ClaudeStalkers
!pip install -q "trl>=0.15.0" "peft>=0.14.0" "transformers>=4.46.0" datasets accelerate bitsandbytes
!pip install -q fastapi pydantic uvicorn openai matplotlib rich
!pip install -e . --quiet

In [ ]:
# Cell 2: Verify environment works
import sys; sys.path.insert(0, '.')
from round2.war_room.environment import WarRoomEnvironment
from round2.war_room.models import MultiAgentAction, AgentAction

env = WarRoomEnvironment()
obs = env.reset('task1', seed=42)
print('✅ Environment reset OK')
print(f'Task: {obs.metadata["task_id"]}, Max rounds: {obs.metadata["max_rounds"]}')
print(f'Triage sees: {obs.triage.text[:100]}...')

In [ ]:
# Cell 3: Run before/after demo (no GPU needed, <1 sec)
!PYTHONPATH=. python round2/war_room/demo_comparison.py

In [ ]:
# Cell 4: Quick GRPO training on T4 (~15 min)
# Uses Qwen2.5-1.5B-Instruct in 4-bit — fits on free T4
!PYTHONPATH=. python round2/war_room/train_t4_quick.py

In [ ]:
# Cell 5: Full GRPO training on A100 (~30-60 min)
# Uses Qwen2.5-7B-Instruct with Unsloth 4-bit + LoRA
# Uncomment below when you have A100 credits:

# !pip install -q unsloth
# !PYTHONPATH=. python round2/war_room/train_colab.py --episodes 30 --tasks task1 task2 task3

In [ ]:
# Cell 6: Generate training curves
!PYTHONPATH=. python round2/war_room/generate_charts.py --output-dir outputs/war_room_grpo

In [ ]:
# Cell 7: Display results
from IPython.display import Image, display
import os

for img in ['baseline_vs_trained.png', 'training_curves.png']:
    path = f'outputs/war_room_grpo/{img}'
    if os.path.exists(path):
        print(f'\n📊 {img}')
        display(Image(filename=path))
    else:
        print(f'⚠️ {img} not found (run training first)')

In [ ]:
# Cell 8: Download results
from google.colab import files
import os

for f in ['outputs/war_room_grpo/metrics.json',
          'outputs/war_room_grpo/training_curves.png',
          'outputs/war_room_grpo/baseline_vs_trained.png']:
    if os.path.exists(f):
        files.download(f)
        print(f'✅ Downloaded {f}')